In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split



   user_id  movie_id  rating  timestamp
0      196       242       3  881250949
1      186       302       3  891717742
2       22       377       1  878887116
3      244        51       2  880606923
4      166       346       1  886397596
(100000, 4)


In [ ]:
ratings = pd.read_csv(
    'ml-100k/u.data',
    sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

# Remap IDs to start at 0
user_map  = {u: i for i, u in enumerate(ratings['user_id'].unique())}
movie_map = {m: i for i, m in enumerate(ratings['movie_id'].unique())}

ratings['user_idx']  = ratings['user_id'].map(user_map)
ratings['movie_idx'] = ratings['movie_id'].map(movie_map)

n_users  = len(user_map)
n_movies = len(movie_map)

print(f"Users: {n_users}, Movies: {n_movies}")

# Normalize ratings to 0-1 range
ratings['rating_norm'] = (ratings['rating'] - 1) / 4.0

# Train/test split
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

Users: 943, Movies: 1682


In [3]:
class MovieDataset(Dataset):
    def __init__(self, df):
        self.users  = torch.tensor(df['user_idx'].values,   dtype=torch.long)
        self.movies = torch.tensor(df['movie_idx'].values,  dtype=torch.long)
        self.ratings = torch.tensor(df['rating_norm'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.users[idx], self.movies[idx], self.ratings[idx]


train_dataset = MovieDataset(train_df)
test_dataset  = MovieDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

In [4]:
class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, n_factors=64, dropout=0.1):
        super().__init__()
        
        # Embedding layers — these are the "latent factors"
        self.user_embeddings  = nn.Embedding(n_users,  n_factors)
        self.movie_embeddings = nn.Embedding(n_movies, n_factors)
        
        # Bias terms for each user and movie
        self.user_bias  = nn.Embedding(n_users,  1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        
        self.dropout = nn.Dropout(dropout)
        
        # Initialize weights — important for stable training
        nn.init.xavier_uniform_(self.user_embeddings.weight)
        nn.init.xavier_uniform_(self.movie_embeddings.weight)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self, user, movie):
        user_emb  = self.dropout(self.user_embeddings(user))
        movie_emb = self.dropout(self.movie_embeddings(movie))
        
        # Dot product of user and movie factors
        dot = (user_emb * movie_emb).sum(dim=1)
        
        # Add biases
        bias = self.user_bias(user).squeeze() + self.movie_bias(movie).squeeze()
        
        # Sigmoid to keep output between 0 and 1 (matches normalized ratings)
        return torch.sigmoid(dot + bias)


model = MatrixFactorization(n_users, n_movies, n_factors=64)
print(model)

MatrixFactorization(
  (user_embeddings): Embedding(943, 64)
  (movie_embeddings): Embedding(1682, 64)
  (user_bias): Embedding(943, 1)
  (movie_bias): Embedding(1682, 1)
  (dropout): Dropout(p=0.1, inplace=False)
)


In [5]:
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model     = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
criterion = nn.MSELoss()

def train_epoch(model, loader):
    model.train()
    total_loss = 0
    for users, movies, ratings in loader:
        users, movies, ratings = users.to(device), movies.to(device), ratings.to(device)
        
        optimizer.zero_grad()
        preds = model(users, movies)
        loss  = criterion(preds, ratings)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for users, movies, ratings in loader:
            users, movies, ratings = users.to(device), movies.to(device), ratings.to(device)
            preds     = model(users, movies)
            total_loss += criterion(preds, ratings).item()
    return total_loss / len(loader)

# Training loop
n_epochs = 20
for epoch in range(n_epochs):
    train_loss = train_epoch(model, train_loader)
    test_loss  = evaluate(model, test_loader)
    
    if (epoch + 1) % 5 == 0:
        # Convert MSE back to original rating scale (1-5) for readability
        train_rmse = np.sqrt(train_loss) * 4
        test_rmse  = np.sqrt(test_loss)  * 4
        print(f"Epoch {epoch+1}/{n_epochs} | Train RMSE: {train_rmse:.4f} | Test RMSE: {test_rmse:.4f}")

Epoch 5/20 | Train RMSE: 0.8976 | Test RMSE: 0.9277
Epoch 10/20 | Train RMSE: 0.8108 | Test RMSE: 0.9073
Epoch 15/20 | Train RMSE: 0.7457 | Test RMSE: 0.9045
Epoch 20/20 | Train RMSE: 0.6991 | Test RMSE: 0.9055


In [6]:
movies_df = pd.read_csv(
    'ml-100k/u.item', sep='|',
    names=['movie_id','title','release_date','video_release_date','imdb_url',
           'unknown','Action','Adventure','Animation','Children','Comedy','Crime',
           'Documentary','Drama','Fantasy','Film-Noir','Horror','Musical',
           'Mystery','Romance','Sci-Fi','Thriller','War','Western'],
    encoding='latin-1'
)

def recommend_for_user(user_id, n=10):
    model.eval()
    
    # Get internal user index
    user_idx = user_map[user_id]
    
    # Movies this user has already rated
    rated_movies = set(ratings[ratings['user_id'] == user_id]['movie_id'])
    
    # Predict ratings for all unrated movies
    unrated_idxs = [
        movie_map[mid] for mid in movie_map
        if mid not in rated_movies
    ]
    
    user_tensor  = torch.tensor([user_idx] * len(unrated_idxs), dtype=torch.long).to(device)
    movie_tensor = torch.tensor(unrated_idxs, dtype=torch.long).to(device)
    
    with torch.no_grad():
        preds = model(user_tensor, movie_tensor).cpu().numpy()
    
    # Map back to movie IDs and get top N
    idx_to_movie = {v: k for k, v in movie_map.items()}
    top_indices  = np.argsort(preds)[::-1][:n]
    top_movie_ids = [idx_to_movie[unrated_idxs[i]] for i in top_indices]
    
    result = movies_df[movies_df['movie_id'].isin(top_movie_ids)][['movie_id', 'title']]
    return result.reset_index(drop=True)


# Example
print(recommend_for_user(user_id=1))

   movie_id                                              title
0       285                              Secrets & Lies (1996)
1       302                           L.A. Confidential (1997)
2       318                            Schindler's List (1993)
3       408                              Close Shave, A (1995)
4       430                                   Duck Soup (1933)
5       474  Dr. Strangelove or: How I Learned to Stop Worr...
6       483                                  Casablanca (1942)
7       511                          Lawrence of Arabia (1962)
8       654                                   Chinatown (1974)
9       657                   Manchurian Candidate, The (1962)


In [7]:
# Save
torch.save(model.state_dict(), 'movie_rec_model.pth')

# Load later
model.load_state_dict(torch.load('movie_rec_model.pth'))
model.eval()

MatrixFactorization(
  (user_embeddings): Embedding(943, 64)
  (movie_embeddings): Embedding(1682, 64)
  (user_bias): Embedding(943, 1)
  (movie_bias): Embedding(1682, 1)
  (dropout): Dropout(p=0.1, inplace=False)
)